# Asteroid Collision

# Problem Statement

We are given an array of integers representing asteroids moving in a straight line.

The absolute value represents the asteroid's size.

The sign represents its direction:

```text
Positive → moving right
Negative → moving left
```

All asteroids move at the same speed.

When two asteroids collide:

- The smaller asteroid explodes.
- If both have the same size, both explode.
- Asteroids moving in the same direction never collide.

Return the state of the asteroids after all collisions.

### Input

An integer array:

```text
asteroids
```

### Output

The array after all possible collisions.

### Examples

```text
Input:
[5, 10, -5]

Output:
[5, 10]
```

`10` destroys `-5`.

---

```text
Input:
[8, -8]

Output:
[]
```

Both asteroids have the same size, so both explode.

---

```text
Input:
[10, 2, -5]

Output:
[10]
```

`2` collides with `-5` and explodes.

Then `10` collides with `-5` and destroys it.

---

```text
Input:
[-2, -1, 1, 2]

Output:
[-2, -1, 1, 2]
```

There are no possible collisions.

# Problem Explanation

The important observation is that **not every pair of asteroids can collide**.

A collision can happen only when:

```text
Left asteroid  → right
Right asteroid → left
```

In terms of signs:

```text
positive → negative
```

For example:

```text
[5, -3]
```

These two asteroids move toward each other.

But:

```text
[-5, 3]
```

do not collide because they move away from each other.

Similarly:

```text
[5, 3]
```

do not collide because both move right.

Therefore, when processing a new asteroid, we only need to consider collisions when:

```text
stack[-1] > 0
```

and:

```text
current < 0
```

This makes a Stack a natural choice.

The Stack represents the asteroids that are still alive.

# Why Use a Stack?

Process the asteroids from left to right.

Suppose:

```text
[10, 5, -3]
```

After processing:

```text
10
5
```

the Stack contains:

```text
[10, 5]
```

Now `-3` arrives.

It moves left, so it can collide with the most recent surviving asteroid:

```text
5
```

This is exactly what a Stack gives us:

```text
Last surviving asteroid
        ↓
      Stack top
        ↓
Potential collision
```

If `5` explodes, the next asteroid to consider is:

```text
10
```

which is again the Stack top.

So collisions naturally behave like repeated Stack pops.

# Collision Conditions

A collision happens only when:

```python
stack[-1] > 0 and asteroid < 0
```

Why?

The Stack top is moving right:

```text
→
```

The current asteroid is moving left:

```text
←
```

Therefore they move toward each other.

All other combinations are safe:

```text
positive + positive
positive + positive

negative + negative
negative + negative

negative + positive
←       →
```

Only:

```text
positive + negative
→       ←
```

can collide.

# Collision Outcomes

Suppose:

```text
top = stack[-1]
current = asteroid
```

The current asteroid is negative, so its size is:

```text
abs(current)
```

### Case 1 — Stack Top Is Smaller

```text
top < abs(current)
```

The Stack asteroid explodes.

Therefore:

```python
stack.pop()
```

But the current asteroid may still collide with another asteroid.

So we continue checking.

---

### Case 2 — Same Size

```text
top == abs(current)
```

Both explode.

Therefore:

```python
stack.pop()
```

and:

```text
current asteroid is destroyed
```

---

### Case 3 — Stack Top Is Larger

```text
top > abs(current)
```

The current asteroid explodes.

The Stack asteroid survives.

Therefore:

```text
current asteroid is destroyed
```

and we stop processing that asteroid.

# Algorithm

For every asteroid:

```text
Set current asteroid as alive

While:
    current asteroid is alive
    AND
    stack is not empty
    AND
    stack top is moving right
    AND
    current asteroid is moving left:

        Compare sizes

        If stack top is smaller:
            pop stack
            continue collision

        If sizes are equal:
            pop stack
            destroy current asteroid
            break

        If stack top is larger:
            destroy current asteroid
            break

    If current asteroid is still alive:
        push it
```

The important part is that a surviving negative asteroid can continue colliding with multiple positive asteroids.

In [1]:
class Solution:

    def asteroidCollision(self, asteroids: list[int]) -> list[int]:

        stack = []

        for asteroid in asteroids:

            alive = True

            while (
                alive
                and asteroid < 0
                and stack
                and stack[-1] > 0
            ):

                if stack[-1] < abs(asteroid):
                    stack.pop()

                elif stack[-1] == abs(asteroid):
                    stack.pop()
                    alive = False

                else:
                    alive = False

            if alive:
                stack.append(asteroid)

        return stack

# Dry Run 1

Input:

```text
[5, 10, -5]
```

Start:

```text
Stack = []
```

### Asteroid 5

No collision.

```text
Stack = [5]
```

### Asteroid 10

Both are moving right.

No collision.

```text
Stack = [5, 10]
```

### Asteroid -5

`-5` moves left.

Stack top:

```text
10
```

Collision is possible.

Compare:

```text
10 > 5
```

Therefore `-5` explodes.

```text
Stack = [5, 10]
```

Final:

```text
[5, 10]
```

# Dry Run 2

Input:

```text
[10, 2, -5]
```

### 10

```text
Stack = [10]
```

### 2

Both move right:

```text
Stack = [10, 2]
```

### -5

`-5` can collide with `2`.

Compare:

```text
2 < 5
```

So `2` explodes.

```text
Stack = [10]
```

But `-5` is still alive.

It now collides with:

```text
10
```

Compare:

```text
10 > 5
```

So `-5` explodes.

Final:

```text
[10]
```

# Dry Run 3

Input:

```text
[8, -8]
```

### 8

```text
Stack = [8]
```

### -8

Collision:

```text
8 == |-8|
```

Both explode.

```text
Stack = []
```

Final:

```text
[]
```

# Dry Run 4

Input:

```text
[-2, -1, 1, 2]
```

Process:

```text
-2
```

No asteroid on the right yet.

```text
Stack = [-2]
```

Next:

```text
-1
```

Both move left.

No collision.

```text
Stack = [-2, -1]
```

Next:

```text
1
```

Moves right.

The previous asteroid is moving left, so they are moving away from each other.

No collision.

```text
Stack = [-2, -1, 1]
```

Next:

```text
2
```

Both move right.

No collision.

Final:

```text
[-2, -1, 1, 2]
```

# Important Insight

The condition:

```python
stack[-1] > 0 and asteroid < 0
```

is the heart of the problem.

Do not simply compare every new asteroid with the Stack top.

First ask:

```text
Can these two asteroids actually move toward each other?
```

Only:

```text
positive → negative
```

creates a collision.

This is an example of using the problem's physical constraints to simplify the algorithm.

# Edge Cases

### No Collisions

```text
[1, 2, 3]
```

Output:

```text
[1, 2, 3]
```

---

### All Negative

```text
[-1, -2, -3]
```

Output:

```text
[-1, -2, -3]
```

---

### Equal Collision

```text
[5, -5]
```

Output:

```text
[]
```

---

### Smaller Negative Asteroid

```text
[10, -5]
```

Output:

```text
[10]
```

---

### Larger Negative Asteroid

```text
[5, -10]
```

Output:

```text
[-10]
```

---

### Chain Collision

```text
[10, 2, -5]
```

Output:

```text
[10]
```

The same negative asteroid can collide with multiple positive asteroids.

# Common Mistakes

### Mistake 1 — Comparing Every Pair

We do not need to compare every asteroid with every other asteroid.

Only the nearest surviving asteroid can collide with the current asteroid.

That is why the Stack works.

---

### Mistake 2 — Forgetting Direction

This is wrong:

```python
while stack and ...
```

We must first verify:

```python
stack[-1] > 0 and asteroid < 0
```

---

### Mistake 3 — Stopping After One Collision

Consider:

```text
[10, 2, -5]
```

After `2` explodes, `-5` is still alive.

It must continue toward:

```text
10
```

Therefore the collision check needs a `while` loop.

---

### Mistake 4 — Using `abs()` for Both Sides

Only the negative asteroid needs its magnitude compared:

```python
abs(asteroid)
```

The Stack top is already positive.

---

### Mistake 5 — Pushing a Destroyed Asteroid

If the current asteroid explodes, it must not be added to the Stack.

# Complexity

Let:

```text
N = number of asteroids
```

Every asteroid is:

```text
Pushed at most once
Popped at most once
```

Therefore:

```text
Time → O(N)
```

The Stack can contain all `N` asteroids in the worst case:

```text
Space → O(N)
```

So:

```text
Time  → O(N)
Space → O(N)
```

# Why Is It O(N)?

The nested `while` loop may look like it could make the solution O(N²).

But an asteroid can be popped only once.

For example:

```text
push asteroid
       ↓
survives
       ↓
may later be popped
       ↓
never returns
```

Therefore:

```text
Maximum pushes = N
Maximum pops   = N
```

The total work is:

```text
O(N)
```

This is another example of amortized analysis.

# Comparison

| Approach | Time | Space |
|---|---:|---:|
| Brute Force Simulation | O(N²) or worse | O(N) |
| Stack Simulation | O(N) | O(N) |

The Stack works because only the most recent surviving asteroid can interact with the incoming asteroid.

# Pattern Recognition

This problem does not use the classic:

```text
Next Greater
Previous Smaller
```

pattern.

Instead, recognize:

```text
Objects arrive one by one
+
The latest surviving object may interact with the new object
+
Destroyed objects are removed permanently
```

This strongly suggests:

```text
Stack
```

The pattern is:

```text
Process left → right
       ↓
Keep surviving elements
       ↓
New element interacts with stack top
       ↓
Pop destroyed elements
       ↓
Push survivor
```

This same general idea appears in many simulation problems.

# Takeaway

The Stack represents the asteroids that are still alive.

For every incoming asteroid:

```text
Can it collide with Stack top?
        ↓
positive + negative?
        ↓
Compare sizes
        ↓
Smaller one disappears
        ↓
Continue if necessary
```

The most important condition is:

```python
stack[-1] > 0 and asteroid < 0
```

The most important implementation detail is:

```text
A surviving negative asteroid may cause multiple
collisions, so use a while loop.
```

Complexity:

```text
Time  → O(N)
Space → O(N)
```

The main lesson:

```text
Stack = current surviving state
```

rather than simply:

```text
Stack = previous elements
```